[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-orange?logo=googlecolab)](https://colab.research.google.com/github/YOUR_USERNAME/satellite-change-detection/blob/main/notebooks/08_full_pipeline.ipynb)

# 08 Full Pipeline

Master notebook that ties together threshold tuning, model upgrades, RAPIDS hooks, and full-resolution inference.

**Prerequisites:** Google Colab GPU runtime, LEVIR-CD dataset access, repo files available.

**Expected runtime:** 90-180 minutes depending on how many training runs you execute


## Imports And Shared Setup

This master notebook imports the reusable package modules so all training, evaluation, and inference logic stays synchronized with the GitHub codebase.

In [ ]:
from pathlib import Path
import torch
import pandas as pd
from src.factory import load_config, build_model
from src.train import train_model
from src.dataset import create_dataloaders, find_data_root
from src.evaluate import threshold_sweep
from src.inference import dask_sliding_window_inference
from src.utils import select_device
# Expected output: imports succeed and the notebook is ready for the end-to-end workflow.


## Step 1 ? Threshold Sweep

This cell completes the original missing validation sweep for the contrastive baseline and saves `threshold_sweep.png` and `pr_curve.png` to `/content`.

In [ ]:
baseline_config = load_config(Path('configs/baseline.yaml'))
baseline_config['data']['root'] = '/content/levir_data'
device = select_device('cuda')
_, _, _, _, val_loader, _ = create_dataloaders(baseline_config)
baseline_model = build_model(baseline_config).to(device)
baseline_model.load_state_dict(torch.load('/content/siamese_baseline_best.pth', map_location=device))
thresholds = torch.linspace(0.1, baseline_config['loss']['margin'] * 1.5, 50)
baseline_sweep = threshold_sweep(baseline_model, val_loader, device, thresholds, Path('/content'), margin=float(baseline_config['loss']['margin']))
BEST_THRESHOLD = baseline_sweep['best_threshold']
BEST_THRESHOLD
# Expected output: a baseline best threshold and saved sweep plots.


## Step 2 ? BCE-Dice + Change Head

This cell trains the first segmentation-aligned model and reports its test metrics.

In [ ]:
step2_summary = train_model(Path('configs/full.yaml'))
step2_summary['test_metrics']
# Expected output: improved F1, precision, recall, and IoU relative to the contrastive baseline.


## Step 3 And Step 4 ? ResNet-18 + CBAM

These steps are represented by the final config in this repo. If you want a no-CBAM comparison, duplicate the config and set `model.use_cbam` to `False` before training.

In [ ]:
final_config = load_config(Path('configs/full.yaml'))
final_model = build_model(final_config)
print(type(final_model).__name__, final_config['model']['use_cbam'])
# Expected output: SiameseResNetUNet True


## Step 5 ? RAPIDS Benchmark Hooks

This cell leaves placeholder rows for the benchmark table after you run the RAPIDS notebook on Colab with a GPU runtime.

In [ ]:
benchmark_table = pd.DataFrame([
    {'Operation': 'Load 10 images', 'CPU time': None, 'GPU (RAPIDS) time': None, 'Speedup': None},
    {'Operation': 'Augment one batch', 'CPU time': None, 'GPU (RAPIDS) time': None, 'Speedup': None},
    {'Operation': 'Threshold sweep (val)', 'CPU time': None, 'GPU (RAPIDS) time': None, 'Speedup': None},
    {'Operation': 'Full inference (test)', 'CPU time': None, 'GPU (RAPIDS) time': None, 'Speedup': None},
])
display(benchmark_table)
# Expected output: benchmark table ready for measured timings.


## Step 6 ? Full-Resolution Sliding Window

This cell runs the final model on five full-resolution test samples with weighted Dask-enabled tiling.

In [ ]:
layout = find_data_root(Path('/content/levir_data'))
final_model = build_model(final_config).to(device)
final_model.load_state_dict(torch.load('/content/siamese_cbam_best.pth', map_location=device))
test_a = sorted((layout.root / 'test' / layout.folder_a).glob('*.png'))[:5]
test_b = sorted((layout.root / 'test' / layout.folder_b).glob('*.png'))[:5]
sliding_results = []
from PIL import Image
for path_a, path_b in zip(test_a, test_b):
    sliding_results.append(dask_sliding_window_inference(final_model, Image.open(path_a), Image.open(path_b), device, tile_size=256, overlap=128, threshold=0.5))
print('Sliding-window runs:', len(sliding_results))
# Expected output: five full-resolution probability maps and binary masks.


## Step 7 ? Final Comparison Report

This final cell prints the requested comparison table and a summary card. Replace the `None` fields with the actual metrics from your Colab runs before submission.

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Baseline (contrastive, custom)', 'F1': 0.08, 'Precision': None, 'Recall': None, 'IoU': None},
    {'Model': 'Step 2: BCE-Dice + head', 'F1': None, 'Precision': None, 'Recall': None, 'IoU': None},
    {'Model': 'Step 3: ResNet-18 backbone', 'F1': None, 'Precision': None, 'Recall': None, 'IoU': None},
    {'Model': 'Step 4: + CBAM attention', 'F1': None, 'Precision': None, 'Recall': None, 'IoU': None},
    {'Model': 'Step 6: Full-res sliding win.', 'F1': None, 'Precision': None, 'Recall': None, 'IoU': None},
])
display(comparison)
print('????????????????????????????????????????????????')
print('?  FINAL RESULTS ? Satellite Change Detection  ?')
print('?  Best model: None                       ?')
print('?  Test F1:    None                       ?')
print('?  Precision:  None   Recall: None  ?')
print('?  IoU:        None                       ?')
print('?  Parameters: None                       ?')
print('?  Inference:  None per 1024?1024 pair    ?')
print('????????????????????????????????????????????????')
# Expected output: complete final report table and summary card.


## Notebook Summary

This master notebook walks through the full project arc from threshold tuning to final full-resolution inference, while keeping the implementation aligned with the Python package committed to the repo.